# Toy example: the halo plot of a known interaction

Appendix F of *Exploring the cloud of feature interaction scores in a Rashomon set* (ICLR 2024) uses the model
`f(x1, x2) = x1 + x2 + x1 * x2` evaluated at the single point `x = (1, 1)` with target `y = 3` and squared-error loss, so the reference loss is 0.

A Rashomon-set model multiplies `x1` by `m1` and `x2` by `m2`. Masking one feature alone costs `(2 - 2 m)^2`; masking both costs
`(3 - (m1 m2 + m1 + m2))^2`. If the two features did not interact, the joint loss would be the sum of the single losses, which is what the
*expected* halo (a circle of radius `epsilon`) shows. The *empirical* halo is the actual joint loss. The gap between them is the interaction.

Part 1 solves this analytically with `sympy`; Part 2 obtains the same picture from `RashomonExplainer`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import solve, symbols

from generalized_rashomon_set import RashomonExplainer, plots, utils
from generalized_rashomon_set.plots._style import colors_vis

## Part 1: analytic halo

For each loss budget `t * epsilon` the single-feature mask has two roots (`m+ >= 1` and `m- <= 1`). The budget `epsilon` is split between the
two features in the ways allowed by `delta = 0.1` (`(0.1, 0.9), (0.2, 0.8), ..., (0.9, 0.1)`); each split combined with the four sign choices
gives one joint model. Its loss is drawn in the direction of the split, mirrored into the quadrant chosen by the signs.

In [2]:
x, m1, m2 = symbols('x m1 m2')
epsilon, delta = 0.1, 0.1
single_loss = (2 - 2 * x) ** 2                    # loss when only one feature is masked by x
joint_loss = (3 - (m1 * m2 + m1 + m2)) ** 2      # loss when both are masked
grid = utils.boundary_grid(delta)                # 0, 0.1, ..., 1
# masks[j] = (m+, m-) reaching loss grid[j] * epsilon (at t = 0 the two roots coincide at m = 1)
masks = []
for t in grid:
    roots = sorted((float(r) for r in solve(single_loss - t * epsilon, x)), reverse=True)
    masks.append(roots if len(roots) == 2 else roots * 2)
masks[:3]

[[1.0, 1.0], [1.05, 0.95], [1.0707106781186548, 0.9292893218813453]]

In [3]:
def analytic_halo(radius):
    emp, exp = [], []
    for a in utils.sum_to_one_tuples(2, delta, radius):           # grid-index splits of the budget
        u = np.array(a, float) / np.linalg.norm(a)
        for s in utils.sign_combos(2):                             # 0 -> m+, 1 -> m-
            sign = np.where(np.array(s) == 0, 1.0, -1.0)
            loss = float(joint_loss.subs({m1: masks[a[0]][s[0]], m2: masks[a[1]][s[1]]}))
            emp.append(loss * u * sign)
            exp.append(radius * epsilon * u * sign)
    return np.array(emp), np.array(exp)

def closed(pts):
    pts = pts[np.argsort(np.arctan2(pts[:, 1], pts[:, 0]))]
    return np.vstack([pts, pts[:1]])

fig, ax = plt.subplots(figsize=(6, 6))
for k, radius in enumerate([0.2, 0.4, 0.6, 0.8, 1.0]):
    emp, exp = analytic_halo(radius)
    lt = [0.8, 0.7, 0.6, 0.5, 0.4][k]
    ax.plot(*closed(emp).T, color=colors_vis(0, lt), marker='o', markersize=3, lw=2.5, label='empirical' if k == 0 else None)
    ax.plot(*closed(exp).T, color=colors_vis(1, lt), lw=2.5, label='expected (no interaction)' if k == 0 else None)
ax.set_aspect('equal'); ax.legend(); ax.set_xlabel('loss change along x1'); ax.set_ylabel('loss change along x2')
ax.set_title('analytic halo, radii 0.2 ... 1.0 epsilon');

The empirical halo is larger than the expected circle where both masks move in the same direction (quadrants I and III) and smaller
where they move in opposite directions: the product term `x1 * x2` amplifies or cancels the two single-feature changes.

## Part 2: the same halo from `RashomonExplainer`

The explainer only needs a prediction function, the data and a loss. With a single data point the reference loss is 0, so the tolerance must be
given as an absolute `epsilon`; permutation-based attributions are meaningless for one row (permuting a single row changes nothing), so
`drop_insensitive=False` keeps both features in the search. The halo plot uses only the losses of the joint-masked models.

In [4]:
X = np.array([[1.0, 1.0]])
y = np.array([3.0])
explainer = RashomonExplainer(None, X, y, predict_fn=lambda Z: Z[:, 0] + Z[:, 1] + Z[:, 0] * Z[:, 1],
                              loss='mse', epsilon=epsilon, delta=delta, drop_insensitive=False, random_state=0)
explainer.sample_rashomon_set()
explainer.explain_interactions(order=2, radii=(0.2, 0.4, 0.6, 0.8, 1.0))
explainer.sampled_models().head(6)

,feature,name,boundary,direction,m,loss,loss_diff
0,0,x0,0.0,+,1.0000,0.000000,0.000000
1,0,x0,0.0,-,1.0000,0.000000,0.000000
2,0,x0,0.1,+,1.0500,0.010000,0.010000
3,0,x0,0.1,-,0.9500,0.010000,0.010000
4,0,x0,0.2,+,1.0707,0.019994,0.019994
5,0,x0,0.2,-,0.9293,0.019994,0.019994


In [5]:
# the greedy search recovers the analytic masks to the search resolution (lr * 1e-4)
found = explainer.samples_.masks[:, :, :]            # (feature, boundary, direction)
print('max |m_found - m_analytic| =', np.abs(found[0] - np.array(masks)).max())

max |m_found - m_analytic| = 8.75655532319719e-05


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plots.halo_plot(explainer, (0, 1), ax=axes[0])
axes[0].set_title('RashomonExplainer')
ax = axes[1]
for radius in (0.2, 0.4, 0.6, 0.8, 1.0):
    emp_a, exp_a = analytic_halo(radius)
    emp_p, exp_p = plots.halo_coordinates(explainer.interactions_, (0, 1), radius)
    ax.plot(*closed(emp_a).T, color='black', lw=3, alpha=0.3, label='analytic' if radius == 0.2 else None)
    ax.plot(*closed(emp_p).T, color=colors_vis(0, 0.5), lw=1.5, ls='--', label='explainer' if radius == 0.2 else None)
ax.set_aspect('equal'); ax.legend(); ax.set_title('empirical halos overlaid')
plt.tight_layout()
fig.savefig('x1x2_simple.png', dpi=150, bbox_inches='tight')

## Control: an additive model

For `f(x1, x2) = x1 + x2` the single-point picture does **not** collapse onto the circle: with `d_k = 1 - m_k` the joint squared error is
`(d1 + d2)^2 = d1^2 + d2^2 + 2 d1 d2`, so the squared-error loss itself carries a cross term even though the model is additive.
The cross term is `2 d1 d2 E[x1 x2]` on a data set, which vanishes for independent zero-mean features. On such data the empirical halo of an
additive model sits on the expected circle (within a few percent of `epsilon`, from the finite-sample correlation of the two features and
the search resolution), while the interaction model above does not.

In [7]:
rng = np.random.default_rng(0)
X_add = rng.normal(size=(2000, 2))
additive = RashomonExplainer(None, X_add, X_add[:, 0] + X_add[:, 1], predict_fn=lambda Z: Z[:, 0] + Z[:, 1],
                             loss='mse', epsilon=epsilon, delta=delta, random_state=0)
additive.sample_rashomon_set()
additive.explain_interactions(order=2, radii=(0.5, 1.0))
emp, exp = plots.halo_coordinates(additive.interactions_, (0, 1), 1.0)
print('max |empirical - expected| / epsilon =', np.abs(emp - exp).max() / epsilon)
plots.halo_plot(additive, (0, 1));

max |empirical - expected| / epsilon = 0.029785275172903858
